# Kanıtlı aday grafı: karar pilotu

Bu defter BetaTour çalışmasının bir koşusu değildir; yeni projenin ilk denemesidir.

**Soru.** Bir model seyrek aday listeleri önerir. Listelerin tamlığını bir alt sınır sertifikası kanıtlayabilir mi? Bir kenar ancak onu kullanan her tur bilinen bir turdan kanıtla uzunsa elenir. Kalan kenarlar her optimal turu içeren bir graf oluşturur. Bir düğümün listesi, düğümün kalan bütün kenarlarını içeriyorsa *kanıtla tamdır*.

**Sertifikalar.**

- **C1:** LKH-3'ün düğüm cezalarıyla 1-ağaç sınırı ve bütün kenarların alfa değerleri. Kendi kodumuz, tam sayı aritmetiği.
- **C2:** Concorde'un kesme düzlemli kök LP'si: alt sınır ve `concorde -z` ile yazılan indirgenmiş maliyetler. Yalnızca kanıtlı optimuma karşı denetimleri geçerse kullanılır.

**Örnekler.** Çalışmanın üreticisiyle 20 yeni örnek: 1.000 ve 2.000 düğüm, tekdüze ve kümeli, beşer tane. Ayrıca optimumu bilinen 4 TSPLIB örneği. Concorde her örneği kanıtlı optimuma kadar çözer.

**Karar kuralı (pilottan önce yazıldı).**

- **G1:** C2, kanıtlı optimumu olan her örnekte denetimleri geçer.
- **G2:** C2 ve ÜS = LKH turu ile kalan grafın ortalama derecesi her sentetik grupta en çok 8'dir. Kısmi dökümde karar verilemeyen düğümlerin payı en çok %1 olmalıdır.
- **G3:** BetaTour'un 5'li listesi kanıtla tam olan düğümlerin payında hem LKH'nin alfa-5 listesini hem sayma listesini, her iki boyutta en az 5 puan aşar.

**Çalıştırma.** Ortam olarak L4 GPU + Yüksek RAM (12 vCPU) seçin. Sonra *Çalışma zamanı → Tümünü çalıştır*. İlk hücre Drive'ı bağlar. Sonuçlar `MyDrive/BetaTour/certpilot1` klasörüne yazılır. Oturum düşerse defteri yeniden çalıştırın; biten adımlar atlanır.

**Tahmini süre (12 vCPU).**

| Adım | Süre |
|---|---|
| Kurulum (LKH, Concorde, QSopt, depo, model) | 5-8 dk |
| Öz sınama | 1-2 dk |
| LKH | 2-4 dk |
| BetaTour | ~1 dk |
| Concorde (kök LP ve kanıtlı optimum) | 30-60 dk |
| Analiz ve özet | 2-4 dk |
| **Toplam** | **yaklaşık 45-80 dk** |

In [ ]:
from google.colab import drive; drive.mount("/content/drive")  # noqa: E702
# Kanıtlı aday grafı: karar pilotu. BetaTour çalışmasının bir koşusu değildir; yeni projenin ilk denemesidir.
# Ortam: L4 GPU + Yüksek RAM (12 vCPU, 53 GB) önerilir. GPU yalnızca BetaTour çıkarımı içindir (CPU'da da çalışır);
# Concorde ve LKH süreçleri vCPU sayısı kadar paralel koşar. 2 vCPU'lu standart ortamda süre birkaç kat uzar.
# Tahmini süre (12 vCPU): kurulum 5-8 dk, öz sınama 1-2 dk, LKH 2-4 dk, BetaTour 1 dk, Concorde 30-60 dk, analiz 2-4 dk.
import os

EXPERIMENT_NAME = "certpilot1"
SAVE_TO_DRIVE = True            # sonuçlar Drive'da: MyDrive/BetaTour/certpilot1 (oturum düşerse biten işler atlanır)

# (grup, düğüm sayısı, örnek sayısı, aile, üretici tohumu); tohumlar önceki koşularda kullanılmadı.
SYNTHETIC = [("uni1k", 1000, 5, "uniform", 15101), ("clu1k", 1000, 5, "clustered", 15202),
             ("uni2k", 2000, 5, "uniform", 15303), ("clu2k", 2000, 5, "clustered", 15404)]
TSPLIB_NAMES = ["pr1002", "pcb1173", "rl1304", "u1817"]   # optimumu bilinen; Concorde'un bulduğu optimum buna eşit olmalı
FIRST_TRIAL_SEEDS = list(range(201, 207))   # BetaTour'un okuduğu altı ilk deneme turu (POPMUSIC-5; çalışmadaki gibi)
K_LIST = 5                      # aday listesi uzunluğu
K_GRAPH = 16                    # BetaTour grafiği: 16 en yakın komşu + topluluk kenarları (çalışmadaki gibi)
RANK_DEPTH = 50                 # en yakın komşu ve alfa sıralamalarının uzunluğu
UB_TIME_PER_NODE = 0.01         # üst sınır (ÜS) için LKH koşusu: düğüm başına saniye
K_RC = 200                      # concorde -z: deneyde düğüm başına yazılan en küçük indirgenmiş maliyet sayısı
K_RC_SELF = 60                  # öz sınamada kısmi dökümün sınandığı değer (tam dökümle karşılaştırılır)
RC_TOL = 1e-3                   # indirgenmiş maliyet karşılaştırmalarında pay (maliyet birimi; yazım hassasiyeti eklenir)
CONCORDE_SEED = 99
ROOT_TIMEOUT_S = 1800           # kök LP koşusu için üst süre
FULL_TIMEOUT_S = 2700           # tam çözüm (kanıtlı optimum) için üst süre; aşılırsa örnek optimumsuz raporlanır
SELF_N = 200                    # öz sınama örneğinin düğüm sayısı
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"     # run v27-v29 ile aynı: tsplab, tsp_model, betatour_v2.pt
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
CODE_SHA256_EXPECTED = "618cdd09270ac4473dc87984a5fb1e7fd757a3946ef34d41aaaf35d84304412a"    # sonraki hücrenin yazdığı certpilot.py dosyasının özeti
NOTEBOOK_BUILD = "f4c672d33a857fbd640a860f9d2181d91994cedff7b2b66d89d1dcf2c759da1d"       # bu defterin bütün kod hücrelerinin özeti

# Karar kuralı (pilottan önce yazıldı; README'deki ile aynı):
# G1  C2 sertifikası, kanıtlı optimumu olan her örnekte denetimleri geçer.
# G2  C2 ve ÜS = LKH turu ile kalan grafın ortalama derecesi her sentetik grupta en çok G2_MAX_MEAN_DEGREE
#     (kısmi dökümde karar verilemeyen düğümlerin payı en çok %1).
# G3  5'li listesi kanıtla tam olan düğümlerin payı (C2, ÜS = LKH turu): BetaTour, hem LKH'nin alfa-5 listesini hem
#     sayma listesini her iki boyutta (1.000 ve 2.000 düğüm; tekdüze ve kümeli birlikte) en az G3_MARGIN_PP puan aşar.
G2_MAX_MEAN_DEGREE = 8.0
G3_MARGIN_PP = 5.0

TEST_MODE = os.environ.get("CERTPILOT_TEST") == "1"     # yalnızca yerel öz sınama; Colab'da kapalıdır
if TEST_MODE:
    EXPERIMENT_NAME = "certpilot_selftest"
    SAVE_TO_DRIVE = False
    SYNTHETIC = [("uni1k", 60, 2, "uniform", 15101), ("clu1k", 70, 1, "clustered", 15202),
                 ("uni2k", 80, 1, "uniform", 15303), ("clu2k", 90, 1, "clustered", 15404)]
    TSPLIB_NAMES = ["test_euc60"]
    SELF_N = 40
    K_RC = 30
    K_RC_SELF = 12
    RANK_DEPTH = 30
    ROOT_TIMEOUT_S = FULL_TIMEOUT_S = 600

In [ ]:
%%writefile certpilot.py
"""Certified candidate graphs: decision pilot (not a run of the BetaTour study).

Question. Can a learned model propose sparse candidate lists whose completeness a lower-bound certificate proves? An
edge is eliminated only when every tour that uses it is provably longer than a known tour; the edges that remain form
a graph that contains every optimal tour. A node's candidate list is certified complete when it contains all of the
node's remaining edges.

Certificates.
C1  The 1-tree bound with LKH-3's node penalties (pi) and the alpha-values of all edges of the complete graph, in
    exact integer arithmetic (this module). A tour that uses edge e costs at least W + alpha(e) (in units of
    1/PRECISION), where W is the bound.
C2  Concorde's root LP with cutting planes: its lower bound and the reduced costs that `concorde -z k` writes (the k
    lowest per node). A tour that uses edge e costs at least LB + rc(e) when rc are the reduced costs of a dual
    solution of value LB. Floating point; used only when the checks of `c2_checks` pass against a proven optimal tour.

All functions are pure numpy except the Concorde download, build and run helpers.
"""
from __future__ import annotations

import glob
import hashlib
import json
import os
import re
import shutil
import subprocess
import tarfile
import time
import urllib.request
from pathlib import Path

import numpy as np

VERSION = "certpilot-1"
PRECISION = 100          # LKH-3 default: node penalties are in units of 1/PRECISION of a distance unit
BIG = np.iinfo(np.int64).max // 4

# ---------------------------------------------------------------------------------------------- Concorde build
CONCORDE_SOURCES = {
    "co031219.tgz": ["https://www.math.uwaterloo.ca/tsp/concorde/downloads/codes/src/co031219.tgz",
                     "http://www.math.uwaterloo.ca/tsp/concorde/downloads/codes/src/co031219.tgz"],
    "qsopt.a": ["https://www.math.uwaterloo.ca/~bico/qsopt/beta/codes/PIC/qsopt.PIC.a",
                "http://www.math.uwaterloo.ca/~bico/qsopt/beta/codes/PIC/qsopt.PIC.a"],
    "qsopt.h": ["https://www.math.uwaterloo.ca/~bico/qsopt/beta/codes/PIC/qsopt.h",
                "http://www.math.uwaterloo.ca/~bico/qsopt/beta/codes/PIC/qsopt.h"],
}
CFLAGS = "-O3 -fcommon -fPIC"


def download(urls, dest, timeout=180, min_bytes=200):
    """First URL that answers; returns (url, sha256)."""
    last = None
    for url in urls:
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (certpilot)"})
            with urllib.request.urlopen(request, timeout=timeout) as response, open(dest, "wb") as f:
                shutil.copyfileobj(response, f)
            data = Path(dest).read_bytes()
            if len(data) < min_bytes:
                raise RuntimeError(f"only {len(data)} bytes")
            return url, hashlib.sha256(data).hexdigest()
        except Exception as exc:  # noqa: BLE001
            last = exc
    raise RuntimeError(f"download failed ({urls[0]}): {last}")


def _run(cmd, cwd, env=None, timeout=1800):
    proc = subprocess.run(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                          errors="replace", timeout=timeout)
    return proc.returncode, proc.stdout


def build_concorde(root, jobs=4):
    """Concorde (co031219) with the QSopt LP solver, from the official University of Waterloo pages. Returns
    {'binary', 'sources': {file: {'url', 'sha256'}}, 'cflags', 'attempts'}. A finished build is reused."""
    root = Path(root)
    binary = root / "concorde" / "TSP" / "concorde"
    marker = root / "build.json"
    if binary.exists() and marker.exists():
        info = json.loads(marker.read_text())
        info["binary"] = str(binary)
        return info
    shutil.rmtree(root, ignore_errors=True)
    (root / "qsopt").mkdir(parents=True)
    sources = {"co031219.tgz": download(CONCORDE_SOURCES["co031219.tgz"], root / "co031219.tgz", min_bytes=100000),
               "qsopt.a": download(CONCORDE_SOURCES["qsopt.a"], root / "qsopt" / "qsopt.a", min_bytes=100000),
               "qsopt.h": download(CONCORDE_SOURCES["qsopt.h"], root / "qsopt" / "qsopt.h")}
    with tarfile.open(root / "co031219.tgz") as tar:
        try:
            tar.extractall(root, filter="data")
        except TypeError:
            tar.extractall(root)
    src = root / "concorde"
    assert (src / "configure").exists(), "Concorde source folder is missing"
    qsopt = str((root / "qsopt").resolve())
    attempts = []
    for libs in ("", "-lpthread -lm"):
        env = dict(os.environ, CFLAGS=CFLAGS)
        if libs:
            env["LIBS"] = libs
        code, out = _run(["./configure", f"--with-qsopt={qsopt}"], src, env)
        if code != 0:
            # An old config.sub/config.guess may not know this machine: use the system's newer copies and retry.
            for name in ("config.sub", "config.guess"):
                found = sorted(glob.glob(f"/usr/share/misc/{name}") + glob.glob(f"/usr/share/automake-*/{name}")
                               + glob.glob(f"/usr/share/libtool/build-aux/{name}"))
                if found:
                    shutil.copy2(found[-1], src / name)
            code, out = _run(["./configure", f"--with-qsopt={qsopt}"], src, env)
        attempts.append({"libs": libs, "configure": code})
        if code != 0:
            continue
        code, out = _run(["make", "-j", str(max(1, int(jobs)))], src, env)
        if code != 0 or not binary.exists():
            code, out = _run(["make"], src, env)
        attempts[-1]["make"] = code
        if code == 0 and binary.exists():
            break
    if not binary.exists():
        raise RuntimeError("Concorde build failed:\n" + out[-4000:])
    info = {"sources": {k: {"url": u, "sha256": s} for k, (u, s) in sources.items()}, "cflags": CFLAGS,
            "attempts": attempts}
    marker.write_text(json.dumps(info, indent=1))
    info["binary"] = str(binary)
    return info


# ------------------------------------------------------------------------------------------------ Concorde runs
NUM = r"(-?\d+(?:\.\d*)?(?:[eE][-+]?\d+)?)"
RX = {"lp": re.compile(r"LP Value\s+\d+:\s*" + NUM),
      "exact": re.compile(r"Exact lower bound:\s*" + NUM),
      "final": re.compile(r"Final lower bound\s*" + NUM + r",\s*upper bound\s*" + NUM),
      "optimal": re.compile(r"Optimal Solution:\s*" + NUM),
      "bbnodes": re.compile(r"Number of bbnodes:\s*(\d+)"),
      "time": re.compile(r"Total Running Time:\s*" + NUM)}


def concorde_usage(binary):
    """The help text that concorde prints when it is run without arguments."""
    try:
        proc = subprocess.run([str(binary)], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, errors="replace", timeout=60)
        return proc.stdout
    except subprocess.TimeoutExpired as exc:
        out = exc.stdout
        return (out.decode("latin-1", "replace") if isinstance(out, bytes) else (out or "")) + "\n[timeout]"


def parse_concorde(text):
    lp = [float(x) for x in RX["lp"].findall(text)]
    exact = [float(x) for x in RX["exact"].findall(text)]
    final = [(float(a), float(b)) for a, b in RX["final"].findall(text)]
    optimal = RX["optimal"].findall(text)
    bbnodes = RX["bbnodes"].findall(text)
    total = RX["time"].findall(text)
    return {"lp_last": lp[-1] if lp else None, "lp_count": len(lp), "exact_last": exact[-1] if exact else None,
            "final_lb": final[-1][0] if final else None, "final_ub": final[-1][1] if final else None,
            "optimal": float(optimal[-1]) if optimal else None, "bbnodes": int(bbnodes[-1]) if bbnodes else None,
            "total_time": float(total[-1]) if total else None}


def root_lower_bound(parsed):
    """The smallest of the root LP value, the exact lower bound and the final lower bound that the run printed. A
    smaller value keeps the certificate valid (it can only eliminate fewer edges)."""
    values = [v for v in (parsed.get("lp_last"), parsed.get("exact_last"), parsed.get("final_lb")) if v is not None]
    return min(values) if values else None


def run_concorde(binary, problem_file, workdir, args, timeout):
    """Runs concorde on a copy of the problem named p.tsp in an empty folder (its files are written there)."""
    workdir = Path(workdir)
    shutil.rmtree(workdir, ignore_errors=True)
    workdir.mkdir(parents=True)
    shutil.copy2(problem_file, workdir / "p.tsp")
    cmd = [str(binary)] + [str(a) for a in args] + ["p.tsp"]
    start = time.time()
    try:
        proc = subprocess.run(cmd, cwd=workdir, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, errors="replace", timeout=timeout)
        text, code, timed_out = proc.stdout, proc.returncode, False
    except subprocess.TimeoutExpired as exc:
        out = exc.stdout
        text = out.decode("latin-1", "replace") if isinstance(out, bytes) else (out or "")
        code, timed_out = None, True
    wall = time.time() - start
    (workdir / "stdout.txt").write_text(text)
    result = {"cmd": [Path(cmd[0]).name] + cmd[1:], "returncode": code, "timed_out": timed_out, "wall": wall,
              "workdir": str(workdir)}
    result.update(parse_concorde(text))
    return result


def read_concorde_tour(path, n):
    """Concorde's tour file: n, then the n nodes (0-based, in input order)."""
    values = np.array(Path(path).read_text().split(), dtype=np.int64)
    assert len(values) >= n + 1 and values[0] == n, "unexpected tour file"
    tour = values[1:n + 1]
    assert np.array_equal(np.sort(tour), np.arange(n)), "tour file is not a permutation"
    return tour


def find_rcn(workdir):
    found = sorted(Path(workdir).glob("*.rcn"))
    return found[0] if found else None


def read_rcn(path, n):
    """Reduced costs written by `concorde -z k`. Accepts an optional 'ncount ecount' header and lines 'u v rc'.
    Returns the distinct undirected edges (key = min*n + max, sorted), their reduced costs (the smaller value when an
    edge is listed twice; the largest difference between two listings is reported), the number of lines and the
    number of listed edges at each node."""
    with open(path) as f:
        head = f.readline().split()
    skip = 1 if len(head) == 2 else 0
    try:
        import pandas as pd
        frame = pd.read_csv(path, sep=r"\s+", header=None, skiprows=skip, usecols=[0, 1, 2],
                            names=["u", "v", "rc"], dtype={"u": np.int64, "v": np.int64, "rc": np.float64})
        u, v, rc = frame["u"].to_numpy(), frame["v"].to_numpy(), frame["rc"].to_numpy()
    except Exception:  # noqa: BLE001
        data = np.loadtxt(path, skiprows=skip, ndmin=2)
        u, v, rc = data[:, 0].astype(np.int64), data[:, 1].astype(np.int64), data[:, 2].astype(np.float64)
    lines = int(len(u))
    if skip:
        assert int(head[0]) == n, f"header says {head[0]} nodes, expected {n}"
    decimals = 0                     # printing precision of the reduced costs (from the first lines)
    with open(path) as f:
        if skip:
            f.readline()
        for _, line in zip(range(5000), f):
            parts = line.split()
            if len(parts) >= 3 and "." in parts[2]:
                decimals = max(decimals, len(parts[2].lower().split("e")[0].split(".")[1]))
    assert lines > 0 and ((u >= 0) & (u < n) & (v >= 0) & (v < n)).all(), "node numbers out of range"
    keep = u != v
    u, v, rc = u[keep], v[keep], rc[keep]
    key = np.minimum(u, v) * n + np.maximum(u, v)
    order = np.lexsort((rc, key))
    key, rc = key[order], rc[order]
    first = np.r_[True, key[1:] != key[:-1]]
    starts = np.flatnonzero(first)
    low = rc[starts]
    high = np.maximum.reduceat(rc, starts)
    keys = key[starts]
    counts = np.bincount(np.r_[keys // n, keys % n], minlength=n)
    return {"keys": keys, "rc": low, "lines": lines, "header": head if skip else None,
            "dup_max_diff": float((high - low).max()) if len(low) else 0.0, "counts": counts,
            "quantum": 10.0 ** (-decimals)}


def node_thresholds(keys, rc, n, k):
    """For a dump that lists the k lowest reduced costs of every node: the k-th lowest value at each node. An edge
    that is not listed has a reduced cost of at least max(thr[u], thr[v]). Nodes with fewer than k listed edges get
    -inf (no bound)."""
    ends = np.r_[keys // n, keys % n]
    values = np.r_[rc, rc]
    order = np.lexsort((values, ends))
    ends, values = ends[order], values[order]
    counts = np.bincount(ends, minlength=n)
    starts = np.searchsorted(ends, np.arange(n))
    thr = np.full(n, -np.inf)
    has = counts >= k
    thr[has] = values[starts[has] + k - 1]
    return thr


def write_full_edge_file(path, D):
    """The complete graph in Concorde's edge-file format: 'n m', then 'u v length' (0-based)."""
    n = len(D)
    iu, ju = np.triu_indices(n, 1)
    with open(path, "w") as f:
        f.write(f"{n} {len(iu)}\n")
        np.savetxt(f, np.column_stack([iu, ju, D[iu, ju]]), fmt="%d")


# ---------------------------------------------------------------------------------------------- edges and tours
def tour_keys(tour, n):
    t = np.asarray(tour, dtype=np.int64)
    a, b = t, np.roll(t, -1)
    return np.sort(np.minimum(a, b) * n + np.maximum(a, b))


def lookup(keys, query):
    """Positions of query in the sorted array keys and a mask of the ones found."""
    if len(keys) == 0:
        return np.zeros(len(query), dtype=np.int64), np.zeros(len(query), dtype=bool)
    pos = np.minimum(np.searchsorted(keys, query), len(keys) - 1)
    return pos, keys[pos] == query


def distance_matrix(xy, ew="EUC_2D"):
    """Integer distances as LKH-3 and Concorde compute them (EUC_2D: nearest integer; CEIL_2D: ceiling)."""
    xy = np.asarray(xy, dtype=np.float64)
    dx = xy[:, None, 0] - xy[None, :, 0]
    dy = xy[:, None, 1] - xy[None, :, 1]
    d = np.sqrt(dx * dx + dy * dy)
    if ew == "EUC_2D":
        return np.floor(d + 0.5).astype(np.int64)
    if ew == "CEIL_2D":
        return np.ceil(d).astype(np.int64)
    raise ValueError(f"unsupported EDGE_WEIGHT_TYPE {ew}")


# ------------------------------------------------------------------------------------------- C1: 1-tree bound
def one_tree(D, pi, precision=PRECISION):
    """Minimum 1-tree of the transformed costs C = precision*D + pi_i + pi_j (LKH-3's form: a minimum spanning tree
    plus, at the leaf whose second-smallest edge is longest, that edge) and the alpha-values of all edges.

    A tour is a 1-tree, so a tour that uses edge e costs at least (L1 + alpha(e) - 2*sum(pi)) / precision, where
    L1 is the cost of the minimum 1-tree and L1 + alpha(e) the cost of the minimum 1-tree that contains e. W =
    L1 - 2*sum(pi) is the bound (all values are integers, in units of 1/precision)."""
    D = np.asarray(D, dtype=np.int64)
    pi = np.asarray(pi, dtype=np.int64)
    n = len(D)
    C = D * precision + pi[:, None] + pi[None, :]
    in_tree = np.zeros(n, dtype=bool)
    in_tree[0] = True
    best = C[0].copy()
    parent = np.zeros(n, dtype=np.int64)
    order = np.zeros(n, dtype=np.int64)
    beta = np.full((n, n), -BIG, dtype=np.int64)  # largest tree edge on the path between two nodes (none: -BIG;
    #                                               transformed costs can be negative, so 0 is not a neutral start)
    for t in range(1, n):
        v = int(np.argmin(np.where(in_tree, BIG, best)))
        p = int(parent[v])
        prev = order[:t]
        row = np.maximum(beta[p, prev], C[p, v])
        beta[v, prev] = row
        beta[prev, v] = row
        order[t] = v
        in_tree[v] = True
        update = (~in_tree) & (C[v] < best)
        best[update] = C[v][update]
        parent[update] = v
    parent[0] = -1
    children = order[1:]
    mst = int(C[children, parent[children]].sum())
    degree = np.bincount(parent[children], minlength=n) + 1
    degree[0] -= 1
    leaves = np.flatnonzero(degree == 1)
    neighbour = parent.copy()
    if degree[0] == 1:
        neighbour[0] = int(np.flatnonzero(parent == 0)[0])
    rows = C[leaves].copy()
    rows[np.arange(len(leaves)), leaves] = BIG
    rows[np.arange(len(leaves)), neighbour[leaves]] = BIG
    second = rows.min(1)
    pick = int(np.argmax(second))
    special, m2 = int(leaves[pick]), int(second[pick])
    alpha = C - beta
    alpha[special, :] = np.maximum(C[special, :] - m2, 0)
    alpha[:, special] = alpha[special, :]
    np.fill_diagonal(alpha, BIG)
    L1 = mst + m2
    return {"W": int(L1 - 2 * int(pi.sum())), "L1": int(L1), "alpha": alpha, "special": special, "mst": mst,
            "precision": int(precision)}


def c1_remaining(tree, ub):
    """Edges that C1 cannot eliminate for an upper bound ub (an integer tour length): W + alpha(e) <= precision*ub.
    Returns the sorted undirected keys."""
    alpha = tree["alpha"]
    n = len(alpha)
    limit = tree["precision"] * int(round(ub)) - tree["W"]
    a, b = np.nonzero(np.triu(alpha <= limit, 1))
    return np.sort(a * n + b)


# -------------------------------------------------------------------------------------- C2: Concorde's root LP
def c2_remaining(keys, rc, lb, ub, n, tol, thr=None):
    """Edges that C2 cannot eliminate for an upper bound ub: listed edges with rc <= ub - lb + tol, and the nodes at
    which unlisted edges might remain (their k-th lowest reduced cost is not above the gap); None for a complete
    dump."""
    gap = ub - lb + tol
    remaining = keys[rc <= gap]
    unresolved = None if thr is None else ~(thr > gap)
    return remaining, unresolved


def c2_checks(keys, rc, lb, n, opt, opt_tour, tol, thr=None, cap=None, sum_negative=0.0, quantum=0.0):
    """Checks of a C2 certificate against a proven optimal tour.
    1. lb <= opt.
    2. No edge of the optimal tour is eliminated (rc > opt - lb + tol).
    3. Dual consistency: opt >= lb + sum of rc over the tour edges + sum of max(0, -rc) over all edges. This holds
       for the reduced costs of a dual solution whose value is at least lb; reduced costs without the cut duals, or
       a bound that belongs to another dual solution, violate it in general.
    keys/rc may hold only the edges with rc <= cap; an edge that is absent then has rc > cap (complete dump) or at
    least max(thr) (partial dump). quantum: printing precision of the reduced costs (each may be off by half of it).
    """
    t_keys = tour_keys(opt_tour, n)
    pos, found = lookup(keys, t_keys)
    a, b = t_keys // n, t_keys % n
    rc_tour = np.where(found, rc[pos], np.nan)
    if thr is None:                       # complete dump: an absent edge has rc > cap (impossible without a cap)
        floor_absent = np.full(len(t_keys), np.inf if cap is None else float(cap))
    else:                                 # partial dump: unlisted (rc >= max thr) or listed above the stored cap
        floor_absent = np.maximum(thr[a], thr[b])
        if cap is not None:
            floor_absent = np.minimum(floor_absent, float(cap))
    gap = opt - lb + tol
    eliminated = (found & (rc_tour > gap)) | (~found & (floor_absent > gap))
    rc_low = np.where(found, rc_tour, np.where(np.isfinite(floor_absent), floor_absent, 0.0))
    slack = float(opt - lb - rc_low.sum() - sum_negative)
    tol_sum = tol * (1.0 + 0.01 * n) + quantum * 2 * n
    out = {"lb_le_opt": bool(lb <= opt + tol), "opt_edges_eliminated": int(eliminated.sum()),
           "opt_edges_unlisted": int((~found).sum()), "dual_slack": slack, "dual_ok": bool(slack >= -tol_sum),
           "root_gap_pct": float(100.0 * (opt - lb) / opt)}
    out["passed"] = bool(out["lb_le_opt"] and out["opt_edges_eliminated"] == 0 and out["dual_ok"])
    return out


# -------------------------------------------------------------------------------------- rankings and measures
def ranking_from_key(key, depth):
    """Per row, the columns with the smallest keys (best first); the diagonal must already be excluded by a large
    key."""
    n = key.shape[0]
    depth = min(depth, n - 1)
    part = np.argpartition(key, depth - 1, axis=1)[:, :depth]
    order = np.argsort(np.take_along_axis(key, part, 1), axis=1, kind="stable")
    return np.take_along_axis(part, order, 1)


def alpha_ranking(alpha, D, depth):
    """All neighbours by alpha-value, ties by distance, then by node number."""
    n = len(D)
    off = ~np.eye(n, dtype=bool)
    if int(alpha[off].max()) < (1 << 29) and int(D.max()) < (1 << 21) and n <= (1 << 12):
        with np.errstate(over="ignore"):
            key = (alpha << 33) + (D << 12) + np.arange(n, dtype=np.int64)[None, :]
        np.fill_diagonal(key, np.iinfo(np.int64).max)
        return ranking_from_key(key, depth)
    depth = min(depth, n - 1)
    out = np.empty((n, depth), dtype=np.int64)
    cols = np.arange(n)
    for i in range(n):
        o = np.lexsort((cols, D[i], alpha[i]))
        out[i] = o[o != i][:depth]
    return out


def nearest_ranking(D, depth):
    n = len(D)
    key = D * n + np.arange(n)[None, :]
    np.fill_diagonal(key, np.iinfo(np.int64).max)
    return ranking_from_key(key, depth)


def certified_k(ranking, remaining, n, unresolved=None):
    """Per node: the shortest prefix of its ranking that contains all of its remaining edges (in the node's own
    list). ranking.shape[1] + 1 when some remaining edge is not in the ranking (or the node is unresolved)."""
    ranking = np.asarray(ranking, dtype=np.int64)
    L = ranking.shape[1]
    rows = np.repeat(np.arange(n, dtype=np.int64), L)
    cols = ranking.ravel()
    valid = cols >= 0
    list_keys = rows[valid] * n + cols[valid]
    list_pos = np.tile(np.arange(1, L + 1, dtype=np.int64), n)[valid]
    order = np.argsort(list_keys)
    list_keys, list_pos = list_keys[order], list_pos[order]
    a, b = remaining // n, remaining % n
    src, dst = np.r_[a, b], np.r_[b, a]
    pos, found = lookup(list_keys, src * n + dst)
    need = np.where(found, list_pos[pos], L + 1)
    k = np.zeros(n, dtype=np.int64)
    np.maximum.at(k, src, need)
    if unresolved is not None:
        k[unresolved] = L + 1
    return k


def edge_coverage(lists, remaining, n):
    """Share of the remaining edges that appear in the list of at least one end."""
    lists = np.asarray(lists, dtype=np.int64)
    L = lists.shape[1]
    rows = np.repeat(np.arange(n, dtype=np.int64), L)
    cols = lists.ravel()
    valid = cols >= 0
    keys = np.unique(np.minimum(rows[valid], cols[valid]) * n + np.maximum(rows[valid], cols[valid]))
    _, found = lookup(keys, remaining)
    return float(found.mean()) if len(remaining) else 1.0


def degree_summary(remaining, n):
    degree = np.bincount(np.r_[remaining // n, remaining % n], minlength=n)
    return {"edges_per_node": float(len(remaining) / n), "mean_degree": float(degree.mean()),
            "median_degree": float(np.median(degree)), "max_degree": int(degree.max()),
            "share_degree_le5": float((degree <= 5).mean()), "share_degree_le8": float((degree <= 8).mean())}


def k_summary(k, L):
    inf = L + 1
    out = {"le5": float((k <= 5).mean()), "not_in_list": float((k >= inf).mean())}
    for m in (8, 10, 16):
        if L >= m:
            out[f"le{m}"] = float((k <= m).mean())
    out["median"] = float(np.median(k)) if (k < inf).mean() > 0.5 else None
    out["p90"] = float(np.quantile(k, 0.9)) if (k < inf).mean() > 0.9 else None
    return out


def auc(score, label):
    """Probability that a random positive scores above a random negative (ties count one half)."""
    from scipy.stats import rankdata
    label = np.asarray(label, dtype=bool)
    n1, n0 = int(label.sum()), int((~label).sum())
    if n1 == 0 or n0 == 0:
        return None
    r = rankdata(score)
    return float((r[label].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def graph_edges(NB, values, n, reduce="max"):
    """Distinct undirected edges of a (n, K) neighbour table with one value per edge (max over both directions)."""
    NB = np.asarray(NB, dtype=np.int64)
    rows = np.repeat(np.arange(n, dtype=np.int64), NB.shape[1])
    cols = NB.ravel()
    vals = np.asarray(values, dtype=np.float64).ravel()
    valid = (cols >= 0) & (cols != rows)
    keys = np.minimum(rows[valid], cols[valid]) * n + np.maximum(rows[valid], cols[valid])
    uniq, inverse = np.unique(keys, return_inverse=True)
    out = np.full(len(uniq), -np.inf)
    np.maximum.at(out, inverse, vals[valid])
    return uniq, out


In [ ]:
# Kurulum (5-8 dk): depo (sabit commit), certpilot.py denetimi, LKH-3.0.14, Concorde + QSopt (resmi kaynak), BetaTour.
import hashlib
import json
import math
import platform
import shutil
import subprocess
import sys
import tarfile
import threading
import time
import traceback
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import torch

CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
if CPU_COUNT < 8 and not TEST_MODE:
    print(f"UYARI: {CPU_COUNT} vCPU görünüyor; Concorde adımı uzun sürer. Yüksek RAM'li L4 ortamı (12 vCPU) önerilir.")


def cpu_model_name():
    path = Path("/proc/cpuinfo")
    if path.exists():
        for line in path.read_text().splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"


WORK_ROOT = Path(os.environ.get("CERTPILOT_WORK", "/content/certpilot_work"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Depo: run v27-v29 ile aynı sabit commit (örnek üreticisi ve LKH yardımcıları: tsplab; BetaTour: tsp_model ve model).
if TEST_MODE and os.environ.get("CERTPILOT_REPO"):
    REPO = Path(os.environ["CERTPILOT_REPO"])
else:
    REPO = WORK_ROOT / "repo"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "--quiet", "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git",
                        str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
    assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M


def checked_module(file_name, expected):
    """Önceki hücrenin yazdığı modül; içerik depodaki certpilot.py ile bayt bayt aynı olmalıdır (dosya sonundaki satır
    sonu sayısı arayüze göre değişebildiği için tek satır sonuna indirilir)."""
    path = Path(file_name).resolve()
    assert path.exists(), f"Önce {file_name} dosyasını yazan hücreyi çalıştırın."
    data = path.read_bytes().rstrip(b"\n") + b"\n"
    digest = hashlib.sha256(data).hexdigest()
    assert digest == expected, f"{file_name} beklenen modülle aynı değil; o hücreyi değiştirmeden çalıştırın."
    if path.read_bytes() != data:
        path.write_bytes(data)
    return path, digest


CODE_PATH, CODE_SHA256 = checked_module("certpilot.py", CODE_SHA256_EXPECTED)
sys.path.insert(0, str(CODE_PATH.parent))
import certpilot as C

if SAVE_TO_DRIVE:
    OUTPUT_ROOT = Path("/content/drive/MyDrive/BetaTour") / EXPERIMENT_NAME
else:
    OUTPUT_ROOT = WORK_ROOT / "results" / EXPERIMENT_NAME
for sub in ["instances", "logs", "summary", "code"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
shutil.copy2(CODE_PATH, OUTPUT_ROOT / "code" / "certpilot.py")


def run_build(cmd, cwd, what):
    build = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=1800)
    if build.returncode != 0:
        raise RuntimeError(what + " derlemesi başarısız:\n" + build.stdout[-3000:])


def get_lkh_3014():
    """Değiştirilmemiş LKH-3.0.14: resmi kaynak; ulaşılamazsa SHA256 denetimli sabit kopya (önceki koşulardaki gibi)."""
    if TEST_MODE and os.environ.get("CERTPILOT_LKH"):
        return os.environ["CERTPILOT_LKH"], "local test binary"
    root = WORK_ROOT / "lkh"
    binary = root / "LKH-3.0.14" / "LKH"
    marker = root / "source.txt"
    if binary.exists() and marker.exists():
        return str(binary), marker.read_text().strip()
    root.mkdir(parents=True, exist_ok=True)
    options = [("official", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
               ("pinned copy", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar",
                "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33", 90)]
    last_error = None
    for label, url, expected_sha, timeout in options:
        archive = root / ("LKH-3.0.14.tar" if url.endswith(".tar") else "LKH-3.0.14.tgz")
        try:
            print("LKH kaynağı:", label, url)
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout) as response, archive.open("wb") as f:
                shutil.copyfileobj(response, f)
            if expected_sha:
                got = hashlib.sha256(archive.read_bytes()).hexdigest()
                assert got == expected_sha, f"LKH kopyası SHA256 uyuşmazlığı: {got}"
            shutil.rmtree(root / "LKH-3.0.14", ignore_errors=True)
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(root, filter="data")
                except TypeError:
                    tar.extractall(root)
            run_build(["make", "-j", str(min(4, CPU_COUNT))], root / "LKH-3.0.14", "LKH")
            assert binary.exists(), "LKH ikilisi oluşmadı"
            marker.write_text(f"{label}: {url}")
            return str(binary), f"{label}: {url}"
        except Exception as exc:
            last_error = exc
            print(f"{label} başarısız: {exc}")
    raise RuntimeError(f"LKH 3.0.14 indirilemedi veya derlenemedi: {last_error}")


t_setup = time.time()
LKH, LKH_SOURCE = get_lkh_3014()
if TEST_MODE and os.environ.get("CERTPILOT_CONCORDE"):
    CONCORDE_INFO = {"binary": os.environ["CERTPILOT_CONCORDE"], "sources": "local test stand-in (not Concorde)"}
else:
    print("Concorde ve QSopt indiriliyor ve derleniyor (2-4 dk)...")
    CONCORDE_INFO = C.build_concorde(WORK_ROOT / "concorde_build", jobs=min(8, CPU_COUNT))
CONCORDE = CONCORDE_INFO["binary"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = REPO / "checkpoints" / "betatour_v2.pt"
assert hashlib.sha256(checkpoint_path.read_bytes()).hexdigest() == CHECKPOINT_SHA256, \
    "Model dosyası beklenen SHA256 değeriyle eşleşmiyor."
MODEL, CK = M.load_model(str(checkpoint_path), DEVICE)      # sabitlenmiş, güvenilen depo
assert MODEL.cfg.get("use_ensemble") is True

HARDWARE = {"device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none",
            "vcpus_visible": CPU_COUNT, "cpu_model": cpu_model_name(), "ram_gb": T.env_info().get("ram_gb"),
            "python": platform.python_version(), "numpy": np.__version__, "torch": torch.__version__}
MANIFEST = {"schema": 1, "experiment_name": EXPERIMENT_NAME, "test_mode": TEST_MODE, "code_version": C.VERSION,
            "code_sha256": CODE_SHA256, "notebook_build": NOTEBOOK_BUILD, "repo_commit": REPO_COMMIT,
            "checkpoint_sha256": CHECKPOINT_SHA256, "lkh_source": LKH_SOURCE,
            "concorde": {k: v for k, v in CONCORDE_INFO.items() if k != "binary"},
            "design": {"synthetic": [list(x) for x in SYNTHETIC], "tsplib": TSPLIB_NAMES,
                       "first_trial_seeds": FIRST_TRIAL_SEEDS, "k_list": K_LIST, "k_graph": K_GRAPH,
                       "rank_depth": RANK_DEPTH, "ub_time_per_node": UB_TIME_PER_NODE, "k_rc": K_RC,
                       "k_rc_self": K_RC_SELF,
                       "rc_tol": RC_TOL, "concorde_seed": CONCORDE_SEED, "root_timeout_s": ROOT_TIMEOUT_S,
                       "full_timeout_s": FULL_TIMEOUT_S, "self_n": SELF_N,
                       "decision": {"g2_max_mean_degree": G2_MAX_MEAN_DEGREE, "g3_margin_pp": G3_MARGIN_PP}},
            "hardware": HARDWARE}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in ["schema", "design", "code_sha256", "repo_commit", "checkpoint_sha256"]:
        assert saved[key] == MANIFEST[key], f"{key} değişti. Ayrı bir EXPERIMENT_NAME kullanın."
    if saved["hardware"] != HARDWARE:
        print("UYARI: donanım ilk oturumdan farklı; süre ölçümleri oturumlar arasında karşılaştırılamaz.")
else:
    manifest_path.write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))

print(f"Hazır ({time.time() - t_setup:.0f} s): {CPU_COUNT} vCPU | {HARDWARE['cpu_model']} | aygıt: {DEVICE} | LKH: "
      f"{LKH_SOURCE} | Concorde: {CONCORDE} | kod {C.VERSION} {CODE_SHA256[:12]} | sonuçlar: {OUTPUT_ROOT}")

In [ ]:
# İşlevler: örnekler, LKH turları ve listeleri, BetaTour, Concorde (kök LP ve kanıtlı optimum), örnek başına analiz.
# Her adım sonucunu Drive'a yazar; oturum düşerse defter yeniden çalıştırıldığında biten adımlar atlanır.
PRINT_LOCK = threading.Lock()
T0 = time.time()


def say(*parts):
    with PRINT_LOCK:
        print(f"[{time.strftime('%H:%M:%S')} +{(time.time() - T0) / 60:5.1f} dk]", *parts, flush=True)


def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=float))
    os.replace(tmp, path)


def read_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None


def save_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)


def make_instances():
    """Yeni sentetik örnekler (çalışmanın üreticisi) ve optimumu bilinen TSPLIB örnekleri."""
    out = []
    for group, n, count, family, seed in SYNTHETIC:
        rng = np.random.default_rng(seed)
        for i in range(count):
            out.append(dict(group=group, family=family, name=f"cp1_{group}_{i:02d}", n=n,
                            xy=T.to_int_coords(T.gen_instance(n, family, rng)), ew="EUC_2D", known_opt=None,
                            source=None))
    if TEST_MODE:                       # TSPLIB biçimli küçük dosya, ondalıklı koordinatlarla; ağ gerekmez
        folder = WORK_ROOT / "tsplib_test"
        folder.mkdir(exist_ok=True)
        paths = {}
        for name in TSPLIB_NAMES:
            count = int(name[len("test_euc"):])
            xy = np.random.default_rng(73 + count).random((count, 2)) * 5000.0
            paths[name] = str(folder / (name + ".tsp"))
            with open(paths[name], "w") as f:
                f.write(f"NAME : {name}\nTYPE : TSP\nDIMENSION : {count}\nEDGE_WEIGHT_TYPE : EUC_2D\n"
                        "NODE_COORD_SECTION\n")
                f.writelines(f"{i} {x:.3f} {y:.3f}\n" for i, (x, y) in enumerate(xy, 1))
                f.write("EOF\n")
    else:
        paths = T.download_tsplib(str(WORK_ROOT / "tsplib"), names=TSPLIB_NAMES)
        absent = sorted(set(TSPLIB_NAMES) - set(paths))
        if absent:
            print("UYARI: TSPLIB dosyaları indirilemedi, bu örnekler atlanıyor:", absent)
    for name in TSPLIB_NAMES:
        if name not in paths:
            continue
        data = T.read_tsplib(paths[name])
        assert data["edge_weight_type"] == "EUC_2D" and len(data["xy"]) == data["dimension"], name
        out.append(dict(group="tsplib", family="tsplib", name=name, n=int(data["dimension"]), xy=data["xy"],
                        ew="EUC_2D", known_opt=T.TSPLIB_OPT.get(name), source=paths[name]))
    problems = WORK_ROOT / "problems"
    problems.mkdir(exist_ok=True)
    for inst in out:
        inst["dir"] = OUTPUT_ROOT / "instances" / inst["name"]
        inst["dir"].mkdir(parents=True, exist_ok=True)
        pf = problems / f"{inst['name']}.tsp"
        if inst["source"] is None:
            T.write_tsp(pf, inst["xy"], name=inst["name"], ew="EUC_2D")
        else:
            shutil.copy2(inst["source"], pf)
        inst["problem_file"] = str(pf)
        digest = hashlib.sha256(pf.read_bytes()).hexdigest()
        meta = inst["dir"] / "instance.json"
        if meta.exists():
            assert read_json(meta)["problem_sha256"] == digest, f"{inst['name']}: örnek dosyası değişti"
        else:
            write_json(meta, dict(name=inst["name"], group=inst["group"], family=inst["family"], n=inst["n"],
                                  known_opt=inst["known_opt"], problem_sha256=digest))
    assert len({x["name"] for x in out}) == len(out)
    return out


# ------------------------------------------------------------------------------------- A: LKH turları ve listeleri
def stage_lkh(inst):
    """Altı ilk deneme turu ve POPMUSIC-5 listeleri (BetaTour'un okuduğu topluluk), LKH'nin alfa-5 listesi ve düğüm
    cezaları (alt gradyan yükselişi; C1 sertifikası bunları kullanır) ve bir üst sınır turu."""
    path = inst["dir"] / "lkh.npz"
    if path.exists():
        return "önbellekte"
    n, xy, pf, ew = inst["n"], inst["xy"], inst["problem_file"], inst["ew"]
    work = WORK_ROOT / "lkh_runs" / inst["name"]
    work.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    firsts = [T.task_first_trial(LKH, str(work), seed, xy, problem_file=pf, ew=ew, ctype="POPMUSIC", k=K_LIST,
                                 timeout=900) for seed in FIRST_TRIAL_SEEDS]
    t1 = time.time()
    al = T.lkh_solve(LKH, problem_file=pf, ew=ew, workroot=str(work), want_cands=True, want_pi=True, timeout=1800,
                     params={"CANDIDATE_SET_TYPE": "ALPHA", "SUBGRADIENT": "YES", "MAX_CANDIDATES": K_LIST,
                             "MAX_TRIALS": 1, "SEED": 1})
    assert "cand" in al and "pi" in al, "LKH aday ya da ceza dosyası yazmadı: " + al["stdout"][-500:]
    t2 = time.time()
    ub = T.lkh_solve(LKH, problem_file=pf, ew=ew, workroot=str(work), timeout=1800,
                     params={"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_CANDIDATES": K_LIST,
                             "MAX_TRIALS": n, "SEED": 1, "TIME_LIMIT": max(5.0, UB_TIME_PER_NODE * n)})
    assert T.is_valid_tour(ub["tour"], n), "üst sınır koşusu geçerli tur vermedi"
    tours = [(T.tour_length(xy, ub["tour"], ew), np.asarray(ub["tour"]))] + [(f["cost"], f["tour"]) for f in firsts]
    ub_cost, ub_tour = min(tours, key=lambda x: x[0])
    t3 = time.time()
    save_npz(path, first_tours=np.stack([f["tour"] for f in firsts]).astype(np.int32),
             first_costs=np.array([f["cost"] for f in firsts]), pop_cand=np.stack([f["cand"] for f in firsts]),
             alpha_cand=T.pad_cols(al["cand"]["cand"], K_LIST), alpha_vals=T.pad_cols(al["cand"]["alpha"], K_LIST),
             pi=np.asarray(al["pi"], dtype=np.int64),
             lkh_lb=np.float64(al["lower_bound"] if al["lower_bound"] is not None else np.nan),
             ub_tour=np.asarray(ub_tour, dtype=np.int32), ub_cost=np.float64(ub_cost),
             times=np.array([t1 - t0, t2 - t1, t3 - t2]))
    shutil.rmtree(work, ignore_errors=True)
    return f"LKH {t3 - t0:.0f} s"


# ------------------------------------------------------------------------------------------------- B: BetaTour
def stage_model(inst):
    path = inst["dir"] / "model.npz"
    if path.exists():
        return "önbellekte"
    L = np.load(inst["dir"] / "lkh.npz")
    succ = np.stack([T.tour_succ_pred(t)[0] for t in L["first_tours"]], axis=1).astype(np.int32)
    pred = M.predict_tourfold(MODEL, inst["xy"], K_GRAPH, DEVICE, ens_succ=succ,
                              ens_cost=L["first_costs"].astype(np.float64))
    save_npz(path, NB=np.asarray(pred["NB"], dtype=np.int32), prob=np.asarray(pred["prob"], dtype=np.float32),
             rank=np.asarray(pred["rank"], dtype=np.int32), t_prep=np.float64(pred["t_prep"]),
             t_model=np.float64(pred["t_model"]))
    return f"BetaTour {pred['t_total']:.1f} s"


# ------------------------------------------------------------------------------------------------- C: Concorde
def full_edge_file(inst):
    path = WORK_ROOT / "full_edges" / f"{inst['name']}.edg"
    if not path.exists():
        path.parent.mkdir(exist_ok=True)
        C.write_full_edge_file(path, C.distance_matrix(inst["xy"], inst["ew"]))
    return path


def concorde_root(inst, z, full_e, tag):
    """Kök LP (dallanmadan; -B), z > 0 ise düğüm başına en küçük z indirgenmiş maliyet dökümüyle (-z z)."""
    args = ["-B", "-x", "-s", CONCORDE_SEED]
    if z:
        args += ["-z", int(z)]
    if full_e:
        args += ["-E", str(full_edge_file(inst))]
    wd = WORK_ROOT / "concorde_runs" / inst["name"] / tag
    res = C.run_concorde(CONCORDE, inst["problem_file"], wd, args, ROOT_TIMEOUT_S)
    shutil.copy2(wd / "stdout.txt", OUTPUT_ROOT / "logs" / f"{inst['name']}_{tag}.txt")
    res["lb"] = C.root_lower_bound(res)
    dump = None
    rcn = C.find_rcn(wd) if z else None
    if rcn is not None and res["lb"] is not None:
        dump = C.read_rcn(rcn, inst["n"])
    shutil.rmtree(wd, ignore_errors=True)
    return res, dump


def job_root(inst):
    path = inst["dir"] / "concorde_root.json"
    if path.exists():
        return "önbellekte"
    n = inst["n"]
    ub_lkh = float(np.load(inst["dir"] / "lkh.npz")["ub_cost"])
    z = 0 if C2_MODE is None else (min(K_RC, n - 1) if C2_MODE["partial"] else n - 1)
    res, dump = concorde_root(inst, z, bool(C2_MODE and C2_MODE["full_e"]), "root")
    rec = {k: v for k, v in res.items() if k != "workdir"}
    rec.update(ub_lkh=ub_lkh, z=z, dump=None)
    if dump is not None:
        lb = res["lb"]
        complete = bool(len(dump["keys"]) == n * (n - 1) // 2)
        thr = C.node_thresholds(dump["keys"], dump["rc"], n, z) if not complete else np.full(n, np.inf)
        cap = (ub_lkh - lb) + max(1.0, 1e-6 * abs(lb)) + max(RC_TOL, dump["quantum"])
        keep = dump["rc"] <= cap
        save_npz(inst["dir"] / "rc_keep.npz", keys=dump["keys"][keep], rc=dump["rc"][keep], thr=thr)
        rec["dump"] = dict(lines=dump["lines"], edges=int(len(dump["keys"])), complete=complete,
                           min_count=int(dump["counts"].min()), dup_max_diff=dump["dup_max_diff"],
                           sum_negative=float(np.clip(-dump["rc"], 0, None).sum()),
                           negative=int((dump["rc"] < -RC_TOL).sum()), min_rc=float(dump["rc"].min()), cap=cap,
                           stored=int(keep.sum()), quantum=dump["quantum"])
    write_json(path, rec)
    return f"kök LP {res['wall']:.0f} s" + (" (süre aşıldı)" if res["timed_out"] else "")


def job_full(inst):
    path = inst["dir"] / "concorde_full.json"
    if path.exists():
        return "önbellekte"
    n = inst["n"]
    wd = WORK_ROOT / "concorde_runs" / inst["name"] / "full"
    res = C.run_concorde(CONCORDE, inst["problem_file"], wd, ["-x", "-s", CONCORDE_SEED, "-o", "opt.sol"],
                         FULL_TIMEOUT_S)
    shutil.copy2(wd / "stdout.txt", OUTPUT_ROOT / "logs" / f"{inst['name']}_full.txt")
    rec = {k: v for k, v in res.items() if k != "workdir"}
    rec.update(tour_ok=False, known_opt=inst["known_opt"], matches_known=None)
    sol = wd / "opt.sol"
    if not res["timed_out"] and res["optimal"] is not None and sol.exists():
        tour = C.read_concorde_tour(sol, n)
        length = float(T.tour_length(inst["xy"], tour, inst["ew"]))
        rec.update(tour_length=length, tour_ok=bool(abs(length - res["optimal"]) < 0.5))
        if rec["tour_ok"]:
            save_npz(inst["dir"] / "opt_tour.npz", tour=tour.astype(np.int32))
        if inst["known_opt"] is not None:
            rec["matches_known"] = bool(abs(res["optimal"] - inst["known_opt"]) < 0.5)
    write_json(path, rec)
    shutil.rmtree(wd, ignore_errors=True)
    if rec["tour_ok"]:
        return f"kanıtlı optimum {res['optimal']:.0f} ({res['wall']:.0f} s, {res['bbnodes']} dal düğümü)"
    return f"optimum yok ({'süre aşıldı' if res['timed_out'] else 'hata'}, {res['wall']:.0f} s)"


# -------------------------------------------------------------------------------------- D: örnek başına analiz
def lists_of(inst, L, Mo, rank_alpha, rank_near, D):
    """Beşli listeler ve uzun sıralamalar. Sayma: BetaTour grafındaki kenarlar, altı ilk deneme turunda kullanılma
    sıklığına göre (eşitlikte kısa kenar önce); çalışmadaki sayma kontrolü gibi."""
    n = inst["n"]
    NB, rank_beta = Mo["NB"].astype(np.int64), Mo["rank"].astype(np.int64)
    succ = np.stack([T.tour_succ_pred(t)[0] for t in L["first_tours"]], axis=1)
    freq = T.ensemble_frequency(NB, succ)
    D_nb = D[np.arange(n)[:, None], NB]
    rank_count = np.take_along_axis(NB, np.lexsort((NB, D_nb, -freq), axis=1), 1)
    pop = L["pop_cand"][0].astype(np.int64)
    lists5 = {"nearest": rank_near[:, :5], "alpha_lkh": L["alpha_cand"].astype(np.int64),
              "alpha_ours": rank_alpha[:, :5], "popmusic": pop, "count": rank_count[:, :5],
              "betatour": rank_beta[:, :5], "count_mix": T.mix_candidates(rank_count, pop, 3, 5),
              "betatour_mix": T.mix_candidates(rank_beta, pop, 3, 5)}
    long = {"nearest": rank_near, "alpha_ours": rank_alpha, "count": rank_count, "betatour": rank_beta}
    return lists5, long, NB, freq


def analyse(inst):
    path = inst["dir"] / "analysis.json"
    if path.exists():
        return "önbellekte"
    n, xy, ew = inst["n"], inst["xy"], inst["ew"]
    L = np.load(inst["dir"] / "lkh.npz")
    Mo = np.load(inst["dir"] / "model.npz")
    root = read_json(inst["dir"] / "concorde_root.json") or {}
    full = read_json(inst["dir"] / "concorde_full.json") or {}
    opt = float(full["optimal"]) if full.get("tour_ok") else None
    opt_tour = np.load(inst["dir"] / "opt_tour.npz")["tour"].astype(np.int64) if opt is not None else None
    ub_lkh = float(L["ub_cost"])
    ubs = {"lkh": ub_lkh}
    if opt is not None:
        ubs["opt"] = opt
    t0 = time.time()
    D = C.distance_matrix(xy, ew)
    out = dict(name=inst["name"], group=inst["group"], n=n, opt=opt, known_opt=inst["known_opt"],
               matches_known=full.get("matches_known"), ub_lkh=ub_lkh,
               ub_gap_pct=None if opt is None else 100.0 * (ub_lkh - opt) / opt,
               concorde_full_s=full.get("wall"), concorde_full_timed_out=full.get("timed_out"),
               bbnodes=full.get("bbnodes"), concorde_root_s=root.get("wall"), lkh_times=L["times"].tolist(),
               model_s=float(Mo["t_prep"] + Mo["t_model"]))
    rem, unresolved = {}, {}
    # C1: 1-tree sınırı ve alfa değerleri, tam sayı aritmetiğiyle (LKH'nin düğüm cezalarıyla)
    tree = C.one_tree(D, L["pi"])
    W = tree["W"] / tree["precision"]
    out["c1"] = dict(bound=W, lkh_lower_bound=float(L["lkh_lb"]), gap_pct=None if opt is None else
                     100.0 * (opt - W) / opt)
    for u, ub in ubs.items():
        rem[("c1", u)] = C.c1_remaining(tree, ub)
    if opt is not None:
        out["c1"]["opt_edges_eliminated"] = int((~np.isin(C.tour_keys(opt_tour, n), rem[("c1", "opt")])).sum())
    rank_alpha = C.alpha_ranking(tree["alpha"], D, RANK_DEPTH)
    out["alpha_top5_agreement"] = float(np.mean([set(rank_alpha[v, :5].tolist()) == set(L["alpha_cand"][v].tolist())
                                                 for v in range(n)]))
    del tree
    out["c1"]["time_s"] = time.time() - t0
    # C2: Concorde'un kök LP'si (indirgenmiş maliyetler; denetimler kanıtlı optimuma karşı)
    rk = inst["dir"] / "rc_keep.npz"
    if root.get("dump") and root.get("lb") is not None and rk.exists():
        R = np.load(rk)
        lb = float(root["lb"])
        thr = None if root["dump"]["complete"] else R["thr"]
        quantum = float(root["dump"]["quantum"])
        tol = max(RC_TOL, quantum)
        c2 = dict(lower_bound=lb, gap_pct=None if opt is None else 100.0 * (opt - lb) / opt, dump=root["dump"],
                  tol=tol)
        for u, ub in ubs.items():
            r, un = C.c2_remaining(R["keys"], R["rc"], lb, ub, n, tol, thr=thr)
            rem[("c2", u)], unresolved[("c2", u)] = r, un
            c2[f"unresolved_share_{u}"] = 0.0 if un is None else float(un.mean())
        if opt is not None:
            c2["checks"] = C.c2_checks(R["keys"], R["rc"], lb, n, opt, opt_tour, tol, thr=thr,
                                       cap=root["dump"]["cap"], sum_negative=root["dump"]["sum_negative"],
                                       quantum=quantum)
        out["c2"] = c2
    elif root.get("lb") is not None:
        out["c2"] = dict(lower_bound=float(root["lb"]), gap_pct=None if opt is None else
                         100.0 * (opt - root["lb"]) / opt, dump=None)
    # Listeler: kanıtlı optimumun kaçırılan kenarları ve kanıtla tamlık
    rank_near = C.nearest_ranking(D, RANK_DEPTH)
    lists5, long, NB, freq = lists_of(inst, L, Mo, rank_alpha, rank_near, D)
    if opt is not None:
        out["missing_opt_pct"] = {name: 100.0 * T.candidate_coverage(lst, opt_tour, ks=[5])[5]["missing"] / n
                                  for name, lst in lists5.items()}
    out["certified"] = {}
    for (cert, u), r in rem.items():
        un = unresolved.get((cert, u))
        entry = dict(degree=C.degree_summary(r, n), unresolved_share=0.0 if un is None else float(un.mean()))
        entry["lists5"] = {name: dict(complete_share=float((C.certified_k(lst, r, n, un) <= 5).mean()),
                                      edge_coverage=C.edge_coverage(lst, r, n)) for name, lst in lists5.items()}
        entry["long"] = {name: C.k_summary(C.certified_k(lst, r, n, un), lst.shape[1]) for name, lst in long.items()}
        out["certified"][f"{cert}_{u}"] = entry
    # AUC: BetaTour grafındaki kenarlar arasında kalanları (kanıtla elenemeyen) ayırt etme
    for key in [("c2", "opt"), ("c2", "lkh"), ("c1", "opt"), ("c1", "lkh")]:
        if key in rem:
            break
    edges, prob = C.graph_edges(NB, Mo["prob"], n)
    _, fq = C.graph_edges(NB, freq, n)
    label = np.isin(edges, rem[key])
    dist = D[edges // n, edges % n].astype(np.float64)
    out["auc"] = dict(certificate=f"{key[0]}_{key[1]}", edges=int(len(edges)), remaining=int(label.sum()),
                      betatour=C.auc(prob, label), count=C.auc(fq, label), neg_distance=C.auc(-dist, label))
    out["analysis_s"] = time.time() - t0
    write_json(path, out)
    return f"analiz {out['analysis_s']:.0f} s"


def run_parallel(func, items, workers, label):
    """İşleri paralel yürütür; hatalar kaydedilir, diğer işler sürer."""
    errors = []
    with ThreadPoolExecutor(max_workers=max(1, workers)) as pool:
        futures = {pool.submit(func, item): item for item in items}
        for fut in as_completed(futures):
            item = futures[fut]
            try:
                say(f"{label}: {item['name']}: {fut.result()}")
            except Exception as exc:
                errors.append((item["name"], repr(exc)))
                say(f"{label}: {item['name']}: HATA {exc!r}")
                with open(OUTPUT_ROOT / "logs" / "errors.txt", "a") as f:
                    f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} {label} {item['name']}\n"
                            f"{traceback.format_exc()}\n")
    return errors

In [ ]:
# Öz sınama (1-2 dk): Concorde'un yardım metni ve -z dökümünün anlamı; C1 ve C2 denetimleri küçük bir örnekte.
# C2 yalnızca şu durumda kullanılır: -z dökümü bütün grafı kapsıyor (z = n-1) ve kanıtlı optimuma karşı denetimleri
# geçiyor. Düğüm başına kısmi döküm (z = K_RC) ayrıca doğrulanırsa deneyde o kullanılır (dosyalar küçük kalır).
import re

SELF = {"utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "n": SELF_N}
usage = C.concorde_usage(CONCORDE)
(OUTPUT_ROOT / "logs" / "concorde_usage.txt").write_text(usage)
SELF["usage_lists_z"] = bool(re.search(r"^\s*-z\b", usage, re.M))
sdir = OUTPUT_ROOT / "selfcheck"
sdir.mkdir(exist_ok=True)
for stale in ["lkh.npz", "model.npz", "concorde_full.json", "opt_tour.npz"]:
    (sdir / stale).unlink(missing_ok=True)
(WORK_ROOT / "problems").mkdir(exist_ok=True)
pf_s = WORK_ROOT / "problems" / "selfcheck.tsp"
xy_s = T.to_int_coords(T.gen_instance(SELF_N, "uniform", np.random.default_rng(424242)))
T.write_tsp(pf_s, xy_s, name="selfcheck")
SI = dict(name="selfcheck", group="selfcheck", family="uniform", n=SELF_N, xy=xy_s, ew="EUC_2D", known_opt=None,
          source=None, dir=sdir, problem_file=str(pf_s))
say("öz sınama:", stage_lkh(SI))
say("öz sınama:", job_full(SI))
say("öz sınama:", stage_model(SI))
full_s = read_json(sdir / "concorde_full.json")
assert full_s["tour_ok"], "Concorde öz sınama örneğini çözemedi; günlük: logs/selfcheck_full.txt"
opt_s = float(full_s["optimal"])
tour_s = np.load(sdir / "opt_tour.npz")["tour"].astype(np.int64)
L_s = np.load(sdir / "lkh.npz")
assert opt_s <= float(L_s["ub_cost"]) + 0.5, "Concorde'un optimumu LKH turundan uzun"

# C1: kendi 1-ağaç hesabımız (tam sayı) LKH'nin cezalarıyla; optimal turun hiçbir kenarı elenmemeli.
D_s = C.distance_matrix(xy_s)
tree_s = C.one_tree(D_s, L_s["pi"])
rank_s = C.alpha_ranking(tree_s["alpha"], D_s, 5)
SELF["c1"] = dict(bound=tree_s["W"] / 100.0, lkh_lower_bound=float(L_s["lkh_lb"]), opt=opt_s,
                  opt_edges_eliminated=int((~np.isin(C.tour_keys(tour_s, SELF_N), C.c1_remaining(tree_s, opt_s))).sum()),
                  alpha_top5_agreement=float(np.mean([set(rank_s[v].tolist()) == set(L_s["alpha_cand"][v].tolist())
                                                      for v in range(SELF_N)])))
assert SELF["c1"]["opt_edges_eliminated"] == 0 and SELF["c1"]["bound"] <= opt_s + 1e-9, SELF["c1"]
del tree_s

# C2: -z dökümünün anlamı. Önce z = n-1 (bütün kenarlar); kapsamıyorsa tam kenar dosyasıyla (-E) yeniden.
attempts, full_e_ok, dump_all = [], None, None
for full_e in (False, True):
    res, dump = concorde_root(SI, SELF_N - 1, full_e, "self_all" + ("_E" if full_e else ""))
    a = dict(full_e=full_e, z=SELF_N - 1, lb=res["lb"], returncode=res["returncode"], wall=res["wall"],
             dump=dump is not None)
    if dump is not None:
        complete = bool(len(dump["keys"]) == SELF_N * (SELF_N - 1) // 2)
        chk = C.c2_checks(dump["keys"], dump["rc"], res["lb"], SELF_N, opt_s, tour_s, max(RC_TOL, dump["quantum"]),
                          sum_negative=float(np.clip(-dump["rc"], 0, None).sum()), quantum=dump["quantum"])
        a.update(lines=dump["lines"], edges=int(len(dump["keys"])), complete=complete,
                 min_count=int(dump["counts"].min()), dup_max_diff=dump["dup_max_diff"], quantum=dump["quantum"],
                 checks=chk)
        if complete and chk["passed"]:
            full_e_ok, dump_all = full_e, dump
    attempts.append(a)
    if full_e_ok is not None:
        break
C2_MODE = None
if full_e_ok is not None:
    res, dump = concorde_root(SI, K_RC_SELF, full_e_ok, "self_partial")
    a = dict(full_e=full_e_ok, z=K_RC_SELF, lb=res["lb"], returncode=res["returncode"], wall=res["wall"],
             dump=dump is not None)
    partial_ok = False
    if dump is not None:
        tol_s = max(RC_TOL, dump["quantum"])
        thr = C.node_thresholds(dump["keys"], dump["rc"], SELF_N, K_RC_SELF)
        chk = C.c2_checks(dump["keys"], dump["rc"], res["lb"], SELF_N, opt_s, tour_s, tol_s, thr=thr,
                          sum_negative=float(np.clip(-dump["rc"], 0, None).sum()), quantum=dump["quantum"])
        listed = np.isin(dump_all["keys"], dump["keys"])
        ua, ub_ = dump_all["keys"][~listed] // SELF_N, dump_all["keys"][~listed] % SELF_N
        thresholds_valid = bool((dump_all["rc"][~listed] >= np.maximum(thr[ua], thr[ub_]) - tol_s).all())
        pos, found = C.lookup(dump_all["keys"], dump["keys"])
        same_rc = float(np.abs(dump_all["rc"][pos[found]] - dump["rc"][found]).max()) if found.any() else None
        per_node = bool(dump["counts"].min() >= K_RC_SELF)
        a.update(lines=dump["lines"], edges=int(len(dump["keys"])), min_count=int(dump["counts"].min()),
                 complete=bool(len(dump["keys"]) == SELF_N * (SELF_N - 1) // 2), quantum=dump["quantum"],
                 per_node=per_node, thresholds_valid=thresholds_valid, max_rc_difference_to_full_dump=same_rc,
                 lb_difference_to_full_dump=float(res["lb"] - attempts[-1]["lb"]), checks=chk)
        partial_ok = per_node and thresholds_valid and chk["passed"] and same_rc is not None and same_rc <= tol_s
    attempts.append(a)
    C2_MODE = dict(full_e=bool(full_e_ok), partial=bool(partial_ok))
SELF["c2_attempts"] = attempts
SELF["c2_mode"] = C2_MODE
write_json(sdir / "selfcheck.json", SELF)

print(f"Öz sınama ({SELF_N} düğüm): Concorde optimumu {opt_s:.0f}, LKH turu {float(L_s['ub_cost']):.0f}; "
      f"-z yardım metninde: {'evet' if SELF['usage_lists_z'] else 'HAYIR'}")
print(f"  C1: 1-ağaç sınırı {SELF['c1']['bound']:.2f} (LKH'nin yazdığı {SELF['c1']['lkh_lower_bound']:.2f}); optimal "
      f"turun elenen kenarı: {SELF['c1']['opt_edges_eliminated']}; alfa ilk-5 listeleri LKH ile aynı: "
      f"%{100 * SELF['c1']['alpha_top5_agreement']:.1f}")
for a in attempts:
    chk = a.get("checks") or {}
    print(f"  C2 denemesi z={a['z']}{' -E' if a['full_e'] else ''}: döküm {'var' if a['dump'] else 'YOK'}"
          + (f", {a['edges']} kenar (tam: {a.get('complete')}), düğüm başına en az {a['min_count']}, "
             f"kök farkı %{chk.get('root_gap_pct', float('nan')):.4f}, denetimler: "
             f"{'geçti' if chk.get('passed') else 'GEÇMEDİ'} (ikili gevşeklik {chk.get('dual_slack', float('nan')):.3f}, "
             f"elenen optimal kenar {chk.get('opt_edges_eliminated')})" if a["dump"] else ""))
if C2_MODE is None:
    print("UYARI: C2 kullanılamıyor. Pilot C1 ve kanıtlı optimum kapsamıyla sürer; G1-G3 değerlendirilemez. "
          "Lütfen logs/concorde_usage.txt ve selfcheck/selfcheck.json dosyalarını gönderin.")
else:
    print(f"C2 kipi: {'kısmi döküm (z = K_RC, eşiklerle)' if C2_MODE['partial'] else 'tam döküm (z = n-1)'}"
          f"{', tam kenar dosyasıyla (-E)' if C2_MODE['full_e'] else ''}")

In [ ]:
# Deney (12 vCPU'da 35-70 dk): A) LKH turları ve listeleri, B) BetaTour, C) Concorde (kök LP ve kanıtlı optimum),
# D) örnek başına analiz. Oturum düşerse defteri baştan çalıştırın: biten adımlar Drive'dan okunur ve atlanır.
T0 = time.time()
INSTANCES = make_instances()
groups_seen = list(dict.fromkeys(i["group"] for i in INSTANCES))
say(f"{len(INSTANCES)} örnek: " + ", ".join(f"{g} {sum(i['group'] == g for i in INSTANCES)}" for g in groups_seen))

say("A) LKH: altı ilk deneme turu, alfa listeleri ve cezalar, üst sınır turu (tahmini 2-4 dk)")
run_parallel(stage_lkh, INSTANCES, CPU_COUNT, "A")
READY = [i for i in INSTANCES if (i["dir"] / "lkh.npz").exists()]

say(f"B) BetaTour çıkarımı, {DEVICE} (tahmini 1 dk)")
run_parallel(stage_model, READY, 1, "B")
READY = [i for i in READY if (i["dir"] / "model.npz").exists()]

say(f"C) Concorde: kök LP ve kanıtlı optimum, {max(1, CPU_COUNT - 1)} paralel süreç (tahmini 30-60 dk; uzun işler önce)")
jobs = []
for inst in READY:
    jobs.append(dict(name=inst["name"] + " (tam çözüm)", n=inst["n"], order=0, run=lambda inst=inst: job_full(inst)))
    jobs.append(dict(name=inst["name"] + " (kök LP)", n=inst["n"], order=1, run=lambda inst=inst: job_root(inst)))
jobs.sort(key=lambda j: (-j["n"], j["order"]))
run_parallel(lambda job: job["run"](), jobs, max(1, CPU_COUNT - 1), "C")

say("D) Analiz: C1 ve C2 sertifikaları, listeler, AUC (tahmini 2-4 dk)")
run_parallel(analyse, READY, 1, "D")
done = [i for i in READY if (i["dir"] / "analysis.json").exists()]
say(f"Bitti: {len(done)}/{len(INSTANCES)} örnek analiz edildi; toplam {(time.time() - T0) / 60:.1f} dk. "
    "Hatalar varsa: logs/errors.txt")

In [ ]:
# Özet ve karar (birkaç saniye). Tablolar ekrana yazılır ve summary/ klasörüne kaydedilir (CSV ve JSON).
ROWS = [r for r in (read_json(i["dir"] / "analysis.json") for i in INSTANCES) if r]
SYN = [g for g, *_ in SYNTHETIC]
GROUPS = SYN + (["tsplib"] if any(r["group"] == "tsplib" for r in ROWS) else [])
BY = {g: [r for r in ROWS if r["group"] == g] for g in GROUPS}
LISTS = [("nearest", "en yakın"), ("alpha_lkh", "alfa-5 (LKH)"), ("alpha_ours", "alfa-5 (C1)"),
         ("popmusic", "POPMUSIC-5"), ("count", "sayma-5"), ("betatour", "BetaTour-5"), ("count_mix", "sayma 3+POP"),
         ("betatour_mix", "BetaTour 3+POP")]
LONG = [("nearest", "en yakın"), ("alpha_ours", "alfa (C1)"), ("count", "sayma"), ("betatour", "BetaTour")]
CERTS = [("c1_opt", "C1, ÜS=opt"), ("c1_lkh", "C1, ÜS=LKH"), ("c2_opt", "C2, ÜS=opt"), ("c2_lkh", "C2, ÜS=LKH")]
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def get(r, *path):
    for p in path:
        if r is None:
            return None
        r = r.get(p) if isinstance(r, dict) else None
    return r


def mean_of(rs, *path, scale=1.0):
    vals = [get(r, *path) for r in rs]
    vals = [float(v) for v in vals if v is not None and not (isinstance(v, float) and math.isnan(v))]
    return scale * float(np.mean(vals)) if vals else np.nan


def table(index, columns):
    frame = pd.DataFrame(columns, index=index)
    return frame


def show(title, frame, digits=3):
    print("\n" + title)
    print(frame.round(digits).to_string())


TABLES = {}
TABLES["1_optimum_ve_sinirlar"] = table(GROUPS, {
    "örnek": [len(BY[g]) for g in GROUPS],
    "kanıtlı opt": [sum(r["opt"] is not None for r in BY[g]) for g in GROUPS],
    "Concorde tam (s, ort)": [mean_of(BY[g], "concorde_full_s") for g in GROUPS],
    "Concorde tam (s, en çok)": [max([r["concorde_full_s"] or 0 for r in BY[g]] or [np.nan]) for g in GROUPS],
    "dal düğümü (ort)": [mean_of(BY[g], "bbnodes") for g in GROUPS],
    "kök LP (s, ort)": [mean_of(BY[g], "concorde_root_s") for g in GROUPS],
    "ÜS(LKH)-opt %": [mean_of(BY[g], "ub_gap_pct") for g in GROUPS],
    "opt-C2 kök %": [mean_of(BY[g], "c2", "gap_pct") for g in GROUPS],
    "opt-C1 1-ağaç %": [mean_of(BY[g], "c1", "gap_pct") for g in GROUPS],
    "TSPLIB opt aynı": [f"{sum(bool(r.get('matches_known')) for r in BY[g])}/{sum(r['known_opt'] is not None for r in BY[g])}"
                        for g in GROUPS]})
show("Tablo 1. Kanıtlı optimum ve alt sınırlar (farklar optimuma göre %)", TABLES["1_optimum_ve_sinirlar"], 4)

TABLES["2_kacirilan_opt_kenarlari"] = table(GROUPS, {label: [mean_of(BY[g], "missing_opt_pct", name) for g in GROUPS]
                                                     for name, label in LISTS})
show("Tablo 2. Kanıtlı optimal turun 5'li listelerde bulunmayan kenarları (tur kenarlarının %'si)",
     TABLES["2_kacirilan_opt_kenarlari"])

deg = {}
for cert, label in CERTS:
    deg[f"{label}: ort derece"] = [mean_of(BY[g], "certified", cert, "degree", "mean_degree") for g in GROUPS]
    deg[f"{label}: en büyük"] = [mean_of(BY[g], "certified", cert, "degree", "max_degree") for g in GROUPS]
    deg[f"{label}: derece<=5 %"] = [mean_of(BY[g], "certified", cert, "degree", "share_degree_le5", scale=100)
                                    for g in GROUPS]
    deg[f"{label}: çözümsüz %"] = [mean_of(BY[g], "certified", cert, "unresolved_share", scale=100) for g in GROUPS]
TABLES["3_kalan_graf"] = table(GROUPS, deg)
show("Tablo 3. Kanıtla elenemeyen kenarların grafı (düğüm derecesi; 'çözümsüz': kısmi dökümde karar verilemeyen düğümler)",
     TABLES["3_kalan_graf"].T, 2)

for cert, label in [("c2_lkh", "C2, ÜS = LKH turu"), ("c2_opt", "C2, ÜS = optimum"), ("c1_lkh", "C1, ÜS = LKH turu")]:
    cols = {lab: [mean_of(BY[g], "certified", cert, "lists5", name, "complete_share", scale=100) for g in GROUPS]
            for name, lab in LISTS}
    cols["olası en çok (derece<=5)"] = [mean_of(BY[g], "certified", cert, "degree", "share_degree_le5", scale=100)
                                        for g in GROUPS]
    TABLES[f"4_tam_liste_{cert}"] = table(GROUPS, cols)
    show(f"Tablo 4 ({label}). 5'li listesi kanıtla tam olan düğümler (%): düğümün kalan bütün kenarları listede",
         TABLES[f"4_tam_liste_{cert}"], 1)

cols = {}
for name, lab in LONG:
    cols[f"{lab}: medyan"] = [mean_of(BY[g], "certified", "c2_lkh", "long", name, "median") for g in GROUPS]
    cols[f"{lab}: %90"] = [mean_of(BY[g], "certified", "c2_lkh", "long", name, "p90") for g in GROUPS]
    cols[f"{lab}: <=10 %"] = [mean_of(BY[g], "certified", "c2_lkh", "long", name, "le10", scale=100) for g in GROUPS]
    cols[f"{lab}: listede yok %"] = [mean_of(BY[g], "certified", "c2_lkh", "long", name, "not_in_list", scale=100)
                                     for g in GROUPS]
TABLES["5_gereken_uzunluk"] = table(GROUPS, cols)
show("Tablo 5. Kanıtla tamlık için gereken liste uzunluğu (C2, ÜS = LKH turu; BetaTour ve sayma: 16 kenarlık graf)",
     TABLES["5_gereken_uzunluk"].T, 1)

TABLES["6_auc"] = table(GROUPS, {"sertifika": [",".join(sorted({r["auc"]["certificate"] for r in BY[g]})) for g in GROUPS],
                                 "BetaTour olasılığı": [mean_of(BY[g], "auc", "betatour") for g in GROUPS],
                                 "sayma sıklığı": [mean_of(BY[g], "auc", "count") for g in GROUPS],
                                 "-uzunluk": [mean_of(BY[g], "auc", "neg_distance") for g in GROUPS]})
show("Tablo 6. BetaTour grafındaki kenarlar: kalanları elenenlerden ayırma (AUC; 0,5 = rastgele)", TABLES["6_auc"])

TABLES["7_c2_denetimleri"] = table(GROUPS, {
    "denetim geçen": [f"{sum(bool(get(r, 'c2', 'checks', 'passed')) for r in BY[g])}/"
                      f"{sum(get(r, 'c2', 'checks') is not None for r in BY[g])}" for g in GROUPS],
    "en küçük ikili gevşeklik": [min([get(r, "c2", "checks", "dual_slack") for r in BY[g]
                                      if get(r, "c2", "checks") is not None] or [np.nan]) for g in GROUPS],
    "elenen opt kenarı (C2)": [sum(get(r, "c2", "checks", "opt_edges_eliminated") or 0 for r in BY[g]) for g in GROUPS],
    "elenen opt kenarı (C1)": [sum(get(r, "c1", "opt_edges_eliminated") or 0 for r in BY[g]) for g in GROUPS],
    "alfa-5 uyumu C1/LKH %": [mean_of(BY[g], "alpha_top5_agreement", scale=100) for g in GROUPS]})
show("Tablo 7. Sertifika denetimleri (kanıtlı optimuma karşı)", TABLES["7_c2_denetimleri"], 3)

# Karar
with_opt = [r for r in ROWS if r["opt"] is not None]
g1 = bool(C2_MODE is not None and with_opt and all(bool(get(r, "c2", "checks", "passed")) for r in with_opt))
have_c2 = all("c2_lkh" in r["certified"] for g in SYN for r in BY[g]) and all(BY[g] for g in SYN)
g2_detail = {g: mean_of(BY[g], "certified", "c2_lkh", "degree", "mean_degree") for g in SYN}
g2_unresolved = max(mean_of(BY[g], "certified", "c2_lkh", "unresolved_share") for g in SYN) if have_c2 else np.nan
g2 = bool(have_c2 and all(v <= G2_MAX_MEAN_DEGREE for v in g2_detail.values()) and g2_unresolved <= 0.01)
g3_detail = {}
for size in sorted({n for _, n, *_ in SYNTHETIC}):
    rs = [r for g, n, *_ in SYNTHETIC if n == size for r in BY[g]]
    share = {name: mean_of(rs, "certified", "c2_lkh", "lists5", name, "complete_share", scale=100)
             for name in ("betatour", "alpha_lkh", "count")}
    share["fark"] = share["betatour"] - max(share["alpha_lkh"], share["count"])
    g3_detail[str(size)] = share
g3 = bool(have_c2 and all(d["fark"] >= G3_MARGIN_PP for d in g3_detail.values()))
if not g1:
    verdict = ("C2 sertifikası denetimleri geçmedi ya da kullanılamadı: önce -z dökümünün anlamı çözülmeli. "
               "logs/ klasöründeki concorde_usage.txt ile selfcheck/selfcheck.json dosyalarını gönderin.")
elif not g2:
    verdict = ("Kök LP sertifikası grafı yeterince seyreltmiyor. Kanıtlı aday listeleri için daha güçlü kanıtlar "
               "(ör. Cook vd.'nin yerel eleme yöntemi) gerekir; fikir bu güçteki kanıtla yürümez.")
elif not g3:
    verdict = ("Kanıtlı seyrek graf elde edilebiliyor, ancak BetaTour basit sıralamalardan belirgin biçimde iyi bir "
               "önerici değil. Yöntem öğrenmeden kurulabilir ya da model bu hedefe göre yeniden eğitilmelidir.")
else:
    verdict = ("Devam: kanıtlı seyrek graf elde ediliyor ve BetaTour onu basit sıralamalardan iyi öneriyor. Sonraki "
               "adım: kanıtla tam listelerle LKH araması ve sertifikanın tam sayı aritmetiğiyle bağımsız denetimi.")
DECISION = {"G1_c2_checks": g1, "G2_sparse_graph": g2, "G2_mean_degree": g2_detail,
            "G2_max_unresolved_share": g2_unresolved, "G3_betatour_proposer": g3,
            "G3_shares_pct": g3_detail, "verdict": verdict, "c2_mode": C2_MODE}
print("\nKarar kuralı (pilottan önce yazıldı)")
print(f"  G1 C2 denetimleri: {'GEÇTİ' if g1 else 'GEÇMEDİ'}")
print(f"  G2 kalan grafın ortalama derecesi <= {G2_MAX_MEAN_DEGREE:g} (C2, ÜS = LKH): {'GEÇTİ' if g2 else 'GEÇMEDİ'} "
      + ", ".join(f"{g} {v:.2f}" for g, v in g2_detail.items())
      + f" (çözümsüz düğüm payı en çok %{100 * g2_unresolved:.2f}; sınır %1)")
print(f"  G3 BetaTour-5, alfa-5 ve sayma-5'i en az {G3_MARGIN_PP:g} puan aşar: {'GEÇTİ' if g3 else 'GEÇMEDİ'} "
      + "; ".join(f"{s} düğüm: BetaTour {d['betatour']:.1f}, alfa {d['alpha_lkh']:.1f}, sayma {d['count']:.1f}"
                  for s, d in g3_detail.items()))
print("  Sonuç:", verdict)

out_dir = OUTPUT_ROOT / "summary"
pd.json_normalize(ROWS).to_csv(out_dir / "instances.csv", index=False)
write_json(out_dir / "summary.json", {"decision": DECISION, "tables": {k: v.reset_index().to_dict(orient="records")
                                                                       for k, v in TABLES.items()},
                                      "selfcheck": SELF, "manifest": MANIFEST})
print(f"\nKaydedildi: {out_dir / 'summary.json'} ve {out_dir / 'instances.csv'}")